# BiLSTM Training — Armenian Participle Punctuation

**Purpose:** Retrain BiLSTM with the corrected tokenizer that properly separates Armenian punctuation marks (U+055B–055F, U+0589) before regex tokenization.

**Run on:** MacBook M1 Pro (CPU). BiLSTM is small enough (~45-50M params), no GPU needed.

**Outputs:**
1. `armenian_vocab_v2.json` — new vocabulary
2. `armenian_embeddings_v2.pt` — new embeddings
3. `bilstm_v3_best.pt` — trained checkpoint
4. `bilstm_v3_results.json` — evaluation metrics


In [ ]:
import os
import re
import json
import math
import time
import random
import difflib
from pathlib import Path
from collections import Counter
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import classification_report, f1_score
import optuna

# ─── Configuration ────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [1]:
# Paths 

# Adjust them to match your local directory structure
# See README.md for expected folder layout.
# DATA_DIR = Path("./Data Full/results_clean")
# OLD_VOCAB_PATH = Path("./Data Full/Bi-LSTM Vocab+Embeddings/armenian_vocab.json")
# OLD_EMB_PATH = Path("./Data Full/Bi-LSTM Vocab+Embeddings/armenian_embeddings.pt")
# OUTPUT_DIR = Path("./Weights - train_bilstm_v3")
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# NEW_VOCAB_PATH = OUTPUT_DIR / "armenian_vocab_v2.json"
# NEW_EMB_PATH = OUTPUT_DIR / "armenian_embeddings_v2.pt"
# CHECKPOINT_PATH = OUTPUT_DIR / "bilstm_v3_best.pt"
# RESULTS_PATH = OUTPUT_DIR / "bilstm_v3_results.json"

# Label scheme (4-class, same as v2)
LABEL_MAP = {"O": 0, "COMMA_AFTER": 1, "BUTH_AFTER": 2, "REMOVE_COMMA": 3}
LABEL_LIST = ["O", "COMMA_AFTER", "BUTH_AFTER", "REMOVE_COMMA"]
NUM_CLASSES = 4

# Data splits (same as v2)
TRAIN_WORKERS = [f"worker_{i}_clean.jsonl" for i in [0, 1, 2, 3, 6]]
VAL_WORKERS = [f"worker_{i}_clean.jsonl" for i in [4, 5]]

# Training constants
EMB_DIM = 300
MAX_SEQ_LEN = 256
NEGATIVE_KEEP_RATIO = 0.15

# Device
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("Using MPS (Apple Silicon GPU)")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("Using CPU")

print(f"Output directory: {OUTPUT_DIR}")
print(f"Data directory: {DATA_DIR}")


Using MPS (Apple Silicon GPU)
Output directory: Weights - train_bilstm_v3
Data directory: Data Full/results_clean


## Phase 0: Buth-Fixed Tokenizer and Label Pipeline

The old regex `[\w\u0530-\u058F]+` treated Armenian punctuation (U+055B–055F, U+0589) as word characters because they fall inside the Armenian Unicode block. This in previous bilstm v1, v2 (not included in the files), caused buth (բութ) ՝ to be absorbed into word tokens, making the model nearly blind to BUTH_AFTER.

**Implemented solution:** Pre-separating all Armenian punctuation before regex tokenization.

Later on, the spacing between words and predicted punctuation marks can be simply removed using basic hard-coding.


In [3]:
# ─── Buth-fixed tokenizer ──────────────────────────────────────
ARMENIAN_PUNCT = '\u055b\u055c\u055d\u055e\u055f\u0589'
ARMENIAN_PUNCT_SET = set(ARMENIAN_PUNCT)

def tokenize_armenian(text):
    '''Splits Armenian text into word and punctuation tokens.
    Buth-fixed: pre-separates Armenian punctuation marks that fall
    inside the Unicode Armenian block (U+0530-058F) before regex.'''
    if not isinstance(text, str) or not text.strip():
        return []
    for ch in ARMENIAN_PUNCT:
        text = text.replace(ch, f' {ch} ')
    return re.findall(r'[\w\u0530-\u058F]+|[^\w\s]', text)

def is_punct_token(tok):
    '''Check if a token is punctuation (including Armenian punct).'''
    if len(tok) == 1 and tok in ARMENIAN_PUNCT_SET:
        return True
    return not re.match(r'[\w\u0530-\u058F]', tok)

# ─── 4-class difflib label generation (matches BiLSTM v2) ─────
def generate_labels(orig_tokens, corr_tokens):
    '''Produce per-token 4-class labels by diffing original vs corrected.
    O=0, COMMA_AFTER=1, BUTH_AFTER=2, REMOVE_COMMA=3.
    Rare classes (COMMA_BEFORE, BUTH_BEFORE, REMOVE_BUTH) are left as O.'''
    labels = [0] * len(orig_tokens)
    sm = difflib.SequenceMatcher(None, orig_tokens, corr_tokens)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == 'equal':
            continue
        if tag in ('delete', 'replace'):
            for i in range(i1, i2):
                if orig_tokens[i] == ',':
                    labels[i] = 3  # REMOVE_COMMA
        if tag in ('insert', 'replace'):
            for j in range(j1, j2):
                tok = corr_tokens[j]
                if tok == ',' and i1 > 0:
                    labels[i1 - 1] = 1  # COMMA_AFTER
                elif tok == '\u055d' and i1 > 0:
                    labels[i1 - 1] = 2  # BUTH_AFTER
    return labels

# ─── Extract word-only labels ─────────────────────────────────
def extract_word_labels(orig_tokens, label_ids):
    '''Extract word-only tokens and labels, skipping punctuation.
    REMOVE_COMMA on a punct token → moved to the preceding word,
    BUT only if that word doesn't already carry an insertion label
    (COMMA_AFTER or BUTH_AFTER), which takes priority in replacement cases.'''
    words = []
    word_labels = []
    for i, (tok, lbl) in enumerate(zip(orig_tokens, label_ids)):
        if is_punct_token(tok):
            if lbl == 3 and word_labels:  # REMOVE_COMMA on punct → preceding word
                if word_labels[-1] == 0:  # only if preceding word is still O
                    word_labels[-1] = 3
            continue
        words.append(tok)
        word_labels.append(lbl)
    return words, word_labels

# Quick test
test_result = tokenize_armenian("բառ՝ test")
print(f"Tokenize test: 'բառ՝ test' → {test_result}")
assert '\u055d' in test_result, "Buth should be a separate token!"
print("✓ Buth correctly separated")

Tokenize test: 'բառ՝ test' → ['բառ', '՝', 'test']
✓ Buth correctly separated


## Load Worker Data and Generate Labels

In [5]:
def load_worker_data(worker_files, data_dir):
    '''Load JSONL worker files and generate token-level labels.
    Fields: "original" and "corrected" (per actual worker JSONL format).
    Handles NO_ACTION_NEEDED and ERROR entries.'''
    all_samples = []
    skipped = 0
    for wf in worker_files:
        path = data_dir / wf
        if not path.exists():
            print(f"WARNING: {path} not found, skipping")
            continue
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    skipped += 1
                    continue
                
                original = rec.get("original", "")
                corrected = rec.get("corrected", "")
                
                # Handle special corrected values
                if corrected == "NO_ACTION_NEEDED":
                    corrected = original
                if not isinstance(corrected, str) or corrected in ("", "ERROR"):
                    skipped += 1
                    continue
                if not isinstance(original, str) or not original.strip():
                    skipped += 1
                    continue
                
                orig_tokens = tokenize_armenian(original)
                corr_tokens = tokenize_armenian(corrected)
                if not orig_tokens or not corr_tokens:
                    skipped += 1
                    continue
                
                # Generate 4-class labels directly
                labels = generate_labels(orig_tokens, corr_tokens)
                
                # Extract word-only tokens and labels (skip punct tokens)
                words, word_labels = extract_word_labels(orig_tokens, labels)
                if not words:
                    skipped += 1
                    continue
                
                all_samples.append({"words": words, "labels": word_labels})
    
    print(f"Loaded {len(all_samples)} samples, skipped {skipped}")
    return all_samples

print("Loading training data...")
train_data = load_worker_data(TRAIN_WORKERS, DATA_DIR)
print(f"\nLoading validation data...")
val_data = load_worker_data(VAL_WORKERS, DATA_DIR)

Loading training data...
Loaded 101047 samples, skipped 0

Loading validation data...
Loaded 11026 samples, skipped 0


## Validate Label Distribution (Critical Check)

With the buth fix, we expect:
- BUTH_AFTER in val: ~1,959 (not 42)
- COMMA_AFTER in val: ~1,113 (not 1,141)
- REMOVE_COMMA in val: ~110 (not 1,077)


In [8]:
def count_labels(data, split_name):
    '''Count label distribution across all samples.'''
    label_counts = Counter()
    total_tokens = 0
    for s in data:
        for lbl in s["labels"]:
            label_counts[lbl] += 1
            total_tokens += 1
    
    print(f"\n{split_name} label distribution ({len(data)} sentences, {total_tokens:,} tokens):")
    print(f"{'Label':<20} {'Count':>10} {'%':>8}")
    print("-" * 40)
    for idx, name in enumerate(LABEL_LIST):
        cnt = label_counts.get(idx, 0)
        pct = 100 * cnt / total_tokens if total_tokens > 0 else 0
        print(f"{name:<20} {cnt:>10,} {pct:>7.2f}%")
    return label_counts, total_tokens

train_lbl_counts, train_total = count_labels(train_data, "TRAIN (all)")
val_lbl_counts, val_total = count_labels(val_data, "VAL (all)")

# Validation checks
val_buth = val_lbl_counts.get(2, 0)
val_comma = val_lbl_counts.get(1, 0)
val_remove = val_lbl_counts.get(3, 0)

print(f"\n─── Validation Checks ───")
print(f"BUTH_AFTER in val:    {val_buth:,} (expected ~1,959)")
print(f"COMMA_AFTER in val:   {val_comma:,} (expected ~1,113)")
print(f"REMOVE_COMMA in val:  {val_remove:,} (expected ~110)")
print(f"Total val tokens:     {val_total:,} (expected ~245,144)")

if val_buth < 500:
    print("\nWARNING: BUTH_AFTER count too low! Tokenizer fix may not be applied correctly.")
else:
    print("\n✓ Label distribution looks correct — buth fix is working.")



TRAIN (all) label distribution (101047 sentences, 2,239,185 tokens):
Label                     Count        %
----------------------------------------
O                     2,210,935   98.74%
COMMA_AFTER               9,126    0.41%
BUTH_AFTER               18,297    0.82%
REMOVE_COMMA                827    0.04%

VAL (all) label distribution (11026 sentences, 245,144 tokens):
Label                     Count        %
----------------------------------------
O                       241,960   98.70%
COMMA_AFTER               1,113    0.45%
BUTH_AFTER                1,959    0.80%
REMOVE_COMMA                112    0.05%

─── Validation Checks ───
BUTH_AFTER in val:    1,959 (expected ~1,959)
COMMA_AFTER in val:   1,113 (expected ~1,113)
REMOVE_COMMA in val:  112 (expected ~110)
Total val tokens:     245,144 (expected ~245,144)

✓ Label distribution looks correct — buth fix is working.


## Negative Filtering (15%)

Same as v2: keep only 15% of all-O sentences to reduce class imbalance.


In [11]:
def apply_negative_filtering(data, keep_ratio, seed=42):
    '''Filter out most all-O sentences (negatives).'''
    rng = random.Random(seed)
    positive = []
    negative = []
    for s in data:
        if any(lbl != 0 for lbl in s["labels"]):
            positive.append(s)
        else:
            negative.append(s)
    
    n_keep = int(len(negative) * keep_ratio)
    rng.shuffle(negative)
    kept_neg = negative[:n_keep]
    
    filtered = positive + kept_neg
    rng.shuffle(filtered)
    
    print(f"Positive: {len(positive):,}, Negative: {len(negative):,}")
    print(f"Kept {n_keep:,} negatives ({keep_ratio*100:.0f}%)")
    print(f"Final: {len(filtered):,} samples")
    return filtered

print("Filtering training data...")
train_filtered = apply_negative_filtering(train_data, NEGATIVE_KEEP_RATIO)

print("\nFiltering validation data...")
val_filtered = apply_negative_filtering(val_data, NEGATIVE_KEEP_RATIO)

# Also keep unfiltered val for fair comparison
val_unfiltered = val_data

# Re-count after filtering
print("\n─── After filtering ───")
train_filt_counts, train_filt_total = count_labels(train_filtered, "TRAIN (filtered)")
val_filt_counts, val_filt_total = count_labels(val_filtered, "VAL (filtered)")


Filtering training data...
Positive: 23,954, Negative: 77,093
Kept 11,563 negatives (15%)
Final: 35,517 samples

Filtering validation data...
Positive: 2,639, Negative: 8,387
Kept 1,258 negatives (15%)
Final: 3,897 samples

─── After filtering ───

TRAIN (filtered) label distribution (35517 sentences, 788,037 tokens):
Label                     Count        %
----------------------------------------
O                       759,787   96.42%
COMMA_AFTER               9,126    1.16%
BUTH_AFTER               18,297    2.32%
REMOVE_COMMA                827    0.10%

VAL (filtered) label distribution (3897 sentences, 86,670 tokens):
Label                     Count        %
----------------------------------------
O                        83,486   96.33%
COMMA_AFTER               1,113    1.28%
BUTH_AFTER                1,959    2.26%
REMOVE_COMMA                112    0.13%


## Phase 0A: Rebuild Vocabulary

Build a new vocabulary from the buth-fixed tokenization of all worker files.


In [16]:
def build_vocabulary(data, min_freq=1):
    '''Build vocabulary from word tokens.'''
    word_counts = Counter()
    for s in data:
        word_counts.update(s["words"])
    
    # Special tokens
    vocab = {"<PAD>": 0, "<UNK>": 1}
    idx = 2
    for word, count in word_counts.most_common():
        if count >= min_freq:
            vocab[word] = idx
            idx += 1
    
    return vocab

# Build from ALL training + val data (use unfiltered for full coverage)
all_data = train_data + val_data
print(f"Building vocabulary from {len(all_data):,} sentences...")
vocab = build_vocabulary(all_data, min_freq=1)
VOCAB_SIZE = len(vocab)
print(f"New vocabulary size: {VOCAB_SIZE:,}")

# Save vocabulary
with open(NEW_VOCAB_PATH, 'w', encoding='utf-8') as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)
print(f"Saved to {NEW_VOCAB_PATH}")

# Compare with old vocabulary
if OLD_VOCAB_PATH.exists():
    with open(OLD_VOCAB_PATH, 'r', encoding='utf-8') as f:
        old_vocab = json.load(f)
    old_size = len(old_vocab)
    
    old_set = set(old_vocab.keys())
    new_set = set(vocab.keys())
    shared = old_set & new_set
    only_old = old_set - new_set
    only_new = new_set - old_set
    
    print(f"\nOld vocab: {old_size:,}")
    print(f"New vocab: {VOCAB_SIZE:,}")
    print(f"Shared:    {len(shared):,}")
    print(f"Only old:  {len(only_old):,} (dropped — e.g., 'word՝' compounds)")
    print(f"Only new:  {len(only_new):,} (new — e.g., standalone '՝', '։')")
    
    # Show some examples
    arm_punct_in_new = [t for t in only_new if len(t) == 1 and t in ARMENIAN_PUNCT_SET]
    print(f"\nArmenian punct tokens now in vocab: {arm_punct_in_new}")
    
    buth_compounds = [t for t in only_old if '՝' in t and len(t) > 1][:10]
    print(f"Dropped buth-compounds (sample): {buth_compounds[:5]}")
else:
    print("Old vocabulary not found — building from scratch")
    old_vocab = None

Building vocabulary from 112,073 sentences...
New vocabulary size: 160,293
Saved to Weights - train_bilstm_v3/armenian_vocab_v2.json

Old vocab: 116,900
New vocab: 160,293
Shared:    101,806
Only old:  15,094 (dropped — e.g., 'word՝' compounds)
Only new:  58,487 (new — e.g., standalone '՝', '։')

Armenian punct tokens now in vocab: []
Dropped buth-compounds (sample): ['գրադարան՝', 'վնասելով՝', 'լիտրերով՝', 'կա՝', 'խանը՝']


## Phase 0B: Rebuild Embeddings

- Tokens shared with old vocab: keep their old GloVe-initialized vectors
- New tokens (e.g., standalone `՝`, `։`): random init (scale 0.01)
- Dropped tokens (e.g., `word՝` compounds): discarded


In [19]:
def build_embeddings(vocab, glove_path, old_vocab=None, old_emb_path=None, emb_dim=300):
    '''Build embedding matrix with 3-tier initialization:
    1. GloVe match → use GloVe vector
    2. No GloVe but exists in old vocab → reuse old vector
    3. Neither → random init (0.01 scale)
    '''
    vocab_size = len(vocab)
    embeddings = torch.randn(vocab_size, emb_dim) * 0.01  # default: random
    
    # Load old embeddings for fallback
    old_emb = None
    if old_vocab and old_emb_path and old_emb_path.exists():
        old_emb = torch.load(old_emb_path, map_location='cpu', weights_only=True)
        print(f"Old embeddings loaded: {old_emb.shape}")
    
    # Load GloVe vectors
    print(f"Loading GloVe from {glove_path}...")
    glove_hits = 0
    old_hits = 0
    random_init = 0
    
    glove_vectors = {}
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip().split(' ')
            word = parts[0]
            if word in vocab:
                glove_vectors[word] = torch.tensor([float(x) for x in parts[1:]], dtype=torch.float32)
    
    print(f"GloVe vectors matched to vocab: {len(glove_vectors):,}")
    
    # Assign embeddings with priority: GloVe > old > random
    for token, idx in vocab.items():
        if token in glove_vectors:
            embeddings[idx] = glove_vectors[token]
            glove_hits += 1
        elif old_emb is not None and old_vocab and token in old_vocab:
            old_idx = old_vocab[token]
            if old_idx < old_emb.shape[0]:
                embeddings[idx] = old_emb[old_idx]
                old_hits += 1
            else:
                random_init += 1
        else:
            random_init += 1
    
    # PAD must be zero
    embeddings[0] = torch.zeros(emb_dim)
    
    print(f"\nEmbedding matrix: {embeddings.shape}")
    print(f"  GloVe matched:     {glove_hits:,} ({100*glove_hits/vocab_size:.1f}%)")
    print(f"  Old emb fallback:  {old_hits:,} ({100*old_hits/vocab_size:.1f}%)")
    print(f"  Random init:       {random_init:,} ({100*random_init/vocab_size:.1f}%)")
    return embeddings

GLOVE_PATH = Path("./Tools/Word Embeddings/GloVe_hy_embeddings.txt")

embeddings = build_embeddings(vocab, GLOVE_PATH,
                              old_vocab=old_vocab if OLD_VOCAB_PATH.exists() else None,
                              old_emb_path=OLD_EMB_PATH,
                              emb_dim=EMB_DIM)

torch.save(embeddings, NEW_EMB_PATH)
print(f"\nSaved to {NEW_EMB_PATH}")

Old embeddings loaded: torch.Size([116900, 300])
Loading GloVe from Tools/Word Embeddings/GloVe_hy_embeddings.txt...
GloVe vectors matched to vocab: 103,240

Embedding matrix: torch.Size([160293, 300])
  GloVe matched:     103,240 (64.4%)
  Old emb fallback:  2 (0.0%)
  Random init:       57,051 (35.6%)

Saved to Weights - train_bilstm_v3/armenian_embeddings_v2.pt


## Model Architecture and Dataset

Same `BiLSTMPunctuator` and `PunctuationDataset` as v2 — unchanged.


In [21]:
# ─── Dataset ──────────────────────────────────────────────────
class PunctuationDataset(Dataset):
    def __init__(self, samples, vocab, max_seq_len=256):
        self.data = []
        unk_idx = vocab.get("<UNK>", 1)
        for s in samples:
            ids = [vocab.get(w, unk_idx) for w in s["words"]]
            lbl = s["labels"]
            # Truncate
            ids = ids[:max_seq_len]
            lbl = lbl[:max_seq_len]
            self.data.append((ids, lbl))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        ids, lbl = self.data[idx]
        return torch.tensor(ids, dtype=torch.long), torch.tensor(lbl, dtype=torch.long)

def collate_fn(batch):
    tokens, labels = zip(*batch)
    return (pad_sequence(tokens, batch_first=True, padding_value=0),
            pad_sequence(labels, batch_first=True, padding_value=-100))

# ─── Model ────────────────────────────────────────────────────
class BiLSTMPunctuator(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size, num_layers,
                 dropout, num_classes, pretrained_emb=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        if pretrained_emb is not None:
            self.embedding.weight.data.copy_(pretrained_emb)
            self.embedding.weight.requires_grad = False  # frozen
        self.lstm = nn.LSTM(input_size=emb_dim, hidden_size=hidden_size,
                           num_layers=num_layers, batch_first=True,
                           bidirectional=True,
                           dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * 2, num_classes)
    
    def forward(self, x):
        out, _ = self.lstm(self.embedding(x))
        return self.classifier(self.dropout(out))

print(f"BiLSTMPunctuator ready")
print(f"Vocab size: {VOCAB_SIZE:,}, Embedding dim: {EMB_DIM}, Classes: {NUM_CLASSES}")


BiLSTMPunctuator ready
Vocab size: 160,293, Embedding dim: 300, Classes: 4


## Class Weights

Compare two strategies:
- **Option A:** Cui et al. beta=0.999 (original v2)
- **Option B:** Sqrt inverse frequency (what worked for HyeBERT)

Both are computed; Optuna will try both.


In [25]:
def compute_cui_weights(label_counts, num_classes, beta=0.999):
    '''Cui et al. Class-Balanced Loss. beta=0.999.'''
    weights = []
    for i in range(num_classes):
        n = label_counts.get(i, 1)
        effective_num = (1.0 - beta**n) / (1.0 - beta)
        weights.append(1.0 / effective_num)
    w = torch.tensor(weights, dtype=torch.float32)
    w = w / w.min()  # normalize so minimum = 1.0
    return w

def compute_sqrt_inv_freq_weights(label_counts, num_classes):
    '''Sqrt inverse frequency weights.'''
    total = sum(label_counts.get(i, 1) for i in range(num_classes))
    weights = []
    for i in range(num_classes):
        n = label_counts.get(i, 1)
        weights.append(math.sqrt(total / (num_classes * n)))
    w = torch.tensor(weights, dtype=torch.float32)
    w = w / w.min()
    return w

# Compute on filtered training data
cui_weights = compute_cui_weights(train_filt_counts, NUM_CLASSES, beta=0.999)
sqrt_weights = compute_sqrt_inv_freq_weights(train_filt_counts, NUM_CLASSES)

print("Class weights comparison:")
print(f"{'Class':<20} {'Cui β=0.999':>12} {'Sqrt InvFreq':>14}")
print("-" * 48)
for i, name in enumerate(LABEL_LIST):
    print(f"{name:<20} {cui_weights[i]:>12.4f} {sqrt_weights[i]:>14.4f}")


Class weights comparison:
Class                 Cui β=0.999   Sqrt InvFreq
------------------------------------------------
O                          1.0000         1.0000
COMMA_AFTER                1.0001         9.1244
BUTH_AFTER                 1.0000         6.4440
REMOVE_COMMA               1.7768        30.3105


## Training Functions

In [38]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    n_batches = 0
    for tokens, labels in dataloader:
        tokens, labels = tokens.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(tokens)
        # Reshape for CrossEntropyLoss
        loss = criterion(logits.view(-1, NUM_CLASSES), labels.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1)

def evaluate(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for tokens, labels in dataloader:
            tokens = tokens.to(device)
            logits = model(tokens)
            preds = logits.argmax(dim=-1).cpu()
            
            # Flatten and filter padding (-100)
            for pred_seq, lbl_seq in zip(preds, labels):
                mask = lbl_seq != -100
                all_preds.extend(pred_seq[mask].tolist())
                all_labels.extend(lbl_seq[mask].tolist())
    
    # Macro F1 on all 4 classes
    macro_f1 = f1_score(all_labels, all_preds, labels=list(range(NUM_CLASSES)),
                        average='macro', zero_division=0)
    return macro_f1, all_preds, all_labels

def get_detailed_report(all_labels, all_preds, label_list):
    '''Get classification report as dict.'''
    report = classification_report(all_labels, all_preds,
                                   labels=list(range(len(label_list))),
                                   target_names=label_list,
                                   output_dict=True, zero_division=0)
    return report

print("Training functions ready")


Training functions ready


## Phase 1: Optuna Hyperparameter Search

15 trials × 3 epochs. Same search space as v2.


In [31]:
N_OPTUNA_TRIALS = 25
OPTUNA_EPOCHS = 3

train_dataset = PunctuationDataset(train_filtered, vocab, MAX_SEQ_LEN)
val_dataset_optuna = PunctuationDataset(val_unfiltered, vocab, MAX_SEQ_LEN)

print(f"Train dataset: {len(train_dataset):,} samples (filtered)")
print(f"Val dataset (unfiltered, for Optuna): {len(val_dataset_optuna):,} samples")

def objective(trial):
    hidden = trial.suggest_categorical("hidden_size", [128, 256, 384])
    layers = trial.suggest_int("num_layers", 1, 3)
    drop = trial.suggest_float("dropout", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    bs = trial.suggest_categorical("batch_size", [64, 128])
    weighting = trial.suggest_categorical("weighting", ["cui", "sqrt"])
    
    if weighting == "cui":
        class_weights = cui_weights.to(DEVICE)
    else:
        class_weights = sqrt_weights.to(DEVICE)
    
    train_loader = DataLoader(train_dataset, batch_size=bs, shuffle=True,
                              collate_fn=collate_fn, num_workers=0)
    val_loader = DataLoader(val_dataset_optuna, batch_size=bs, shuffle=False,
                            collate_fn=collate_fn, num_workers=0)
    
    model = BiLSTMPunctuator(vocab_size=VOCAB_SIZE, emb_dim=EMB_DIM, hidden_size=hidden,
        num_layers=layers, dropout=drop, num_classes=NUM_CLASSES,
        pretrained_emb=embeddings).to(DEVICE)
    
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)
    
    best_f1 = 0
    for epoch in range(OPTUNA_EPOCHS):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_f1, _, _ = evaluate(model, val_loader, DEVICE)
        
        trial.report(val_f1, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        
        best_f1 = max(best_f1, val_f1)
    
    return best_f1

print(f"\nReady for Optuna: {N_OPTUNA_TRIALS} trials × {OPTUNA_EPOCHS} epochs")


Train dataset: 35,517 samples (filtered)
Val dataset (unfiltered, for Optuna): 11,026 samples

Ready for Optuna: 25 trials × 3 epochs


## Run Optuna Search

In [34]:
print(f"Starting Optuna search at {datetime.now().strftime('%H:%M:%S')}...")
start_time = time.time()

study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_startup_trials=5))
study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

elapsed = time.time() - start_time
print(f"\nOptuna completed in {elapsed/60:.1f} minutes")
print(f"Best trial: #{study.best_trial.number}")
print(f"Best macro-F1: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

[I 2026-03-30 03:34:42,041] A new study created in memory with name: no-name-c15fdca9-bc45-4224-8b75-11295831bd88


Starting Optuna search at 03:34:42...


  0%|          | 0/25 [00:00<?, ?it/s]

[I 2026-03-30 03:36:03,115] Trial 0 finished with value: 0.34080204463598335 and parameters: {'hidden_size': 256, 'num_layers': 2, 'dropout': 0.4377993180735172, 'lr': 0.0001023154243899605, 'batch_size': 64, 'weighting': 'sqrt'}. Best is trial 0 with value: 0.34080204463598335.
[I 2026-03-30 03:37:58,967] Trial 1 finished with value: 0.3807232569387386 and parameters: {'hidden_size': 256, 'num_layers': 3, 'dropout': 0.14958736359098004, 'lr': 0.00048300897662445486, 'batch_size': 64, 'weighting': 'sqrt'}. Best is trial 1 with value: 0.3807232569387386.
[I 2026-03-30 03:39:19,327] Trial 2 finished with value: 0.38572167591468676 and parameters: {'hidden_size': 256, 'num_layers': 2, 'dropout': 0.20423013199496381, 'lr': 0.0005147887928481624, 'batch_size': 64, 'weighting': 'sqrt'}. Best is trial 2 with value: 0.38572167591468676.
[I 2026-03-30 03:40:26,188] Trial 3 finished with value: 0.3933144570988481 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.3905085291751159

## Phase 2: Final Training with Best Config

10 epochs, patience=2, early stopping on macro-F1.


In [42]:
# Best params from Optuna
best_params = study.best_params
HIDDEN = best_params["hidden_size"]
LAYERS = best_params["num_layers"]
DROPOUT = best_params["dropout"]
LR = best_params["lr"]
BS = best_params["batch_size"]
WEIGHTING = best_params["weighting"]

print(f"Training with best config:")
print(f"  hidden={HIDDEN}, layers={LAYERS}, dropout={DROPOUT:.4f}")
print(f"  lr={LR:.6f}, batch_size={BS}, weighting={WEIGHTING}")

# Class weights
if WEIGHTING == "cui":
    final_weights = cui_weights.to(DEVICE)
    BETA = 0.999
else:
    final_weights = sqrt_weights.to(DEVICE)
    BETA = None

print(f"  Class weights: {final_weights.cpu().tolist()}")

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BS, shuffle=True,
                          collate_fn=collate_fn, num_workers=0)
val_loader_filt = DataLoader(PunctuationDataset(val_filtered, vocab, MAX_SEQ_LEN),
                             batch_size=BS, shuffle=False,
                             collate_fn=collate_fn, num_workers=0)

# Model
model = BiLSTMPunctuator(vocab_size=VOCAB_SIZE, emb_dim=EMB_DIM, hidden_size=HIDDEN,
                        num_layers=LAYERS, dropout=DROPOUT, num_classes=NUM_CLASSES,
                        pretrained_emb=embeddings).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
criterion = nn.CrossEntropyLoss(weight=final_weights, ignore_index=-100)

# Training loop
MAX_EPOCHS = 10
PATIENCE = 2
best_val_f1 = 0
best_epoch = 0
patience_counter = 0
history = []

print(f"\nStarting training at {datetime.now().strftime('%H:%M:%S')}...")
print(f"{'Epoch':>5} {'Train Loss':>12} {'Val F1':>10} {'Best':>6} {'Time':>8}")
print("-" * 45)

training_start = time.time()
for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start = time.time()
    
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_f1, _, _ = evaluate(model, val_loader_filt, DEVICE)
    
    epoch_time = time.time() - epoch_start
    is_best = val_f1 > best_val_f1
    
    if is_best:
        best_val_f1 = val_f1
        best_epoch = epoch
        patience_counter = 0
        # Savimg the best model
        torch.save({
            "model_state_dict": model.state_dict(),
            "vocab_size": VOCAB_SIZE,
            "embedding_dim": EMB_DIM,
            "num_classes": NUM_CLASSES,
            "hyperparameters": best_params,
            "label_map": LABEL_MAP,
            "class_weights": final_weights.cpu().tolist(),
            "beta": BETA,
            "weighting": WEIGHTING,
            "negative_keep_ratio": NEGATIVE_KEEP_RATIO,
            "best_val_f1": best_val_f1,
            "best_epoch": best_epoch,
            "tokenizer_version": "buth_fixed",
            "train_split": "workers_0-3+6",
            "val_split": "workers_4-5"}, CHECKPOINT_PATH)
    else:
        patience_counter += 1
    
    history.append({"epoch": epoch,
               "train_loss": train_loss,
                   "val_f1": val_f1,
                  "is_best": is_best})
    
    marker = " ★" if is_best else ""
    print(f"{epoch:>5} {train_loss:>12.4f} {val_f1:>10.4f} {'★' if is_best else '':>6} {epoch_time:>7.1f}s")
    
    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} (patience={PATIENCE})")
        break

total_time = time.time() - training_start
print(f"\nTraining complete in {total_time/60:.1f} minutes")
print(f"Best epoch: {best_epoch}, Best val F1: {best_val_f1:.4f}")


Training with best config:
  hidden=128, layers=1, dropout=0.3650
  lr=0.004304, batch_size=64, weighting=sqrt
  Class weights: [1.0, 9.124429702758789, 6.44400691986084, 30.310504913330078]

Total parameters: 48,529,248
Trainable parameters: 441,348

Starting training at 03:55:28...
Epoch   Train Loss     Val F1   Best     Time
---------------------------------------------
    1       0.5255     0.4741      ★    10.3s
    2       0.3963     0.4776      ★     9.3s
    3       0.3338     0.4959      ★     9.3s
    4       0.2800     0.5000      ★     8.9s
    5       0.2376     0.4969            9.3s
    6       0.2057     0.5077      ★     9.0s
    7       0.1809     0.5082      ★     9.1s
    8       0.1642     0.5182      ★     9.0s
    9       0.1515     0.5178            9.0s
   10       0.1400     0.5152            8.9s

Early stopping at epoch 10 (patience=2)

Training complete in 1.6 minutes
Best epoch: 8, Best val F1: 0.5182


## Phase 2B: Evaluation on Both Filtered and Unfiltered Val Sets

In [44]:
# Load best model
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model_eval = BiLSTMPunctuator(
    vocab_size=checkpoint["vocab_size"], emb_dim=checkpoint["embedding_dim"],
    hidden_size=best_params["hidden_size"], num_layers=best_params["num_layers"],
    dropout=best_params["dropout"], num_classes=checkpoint["num_classes"]
).to(DEVICE)
model_eval.load_state_dict(checkpoint["model_state_dict"])
model_eval.eval()

# ── Filtered val evaluation ──
print("=" * 60)
print("EVALUATION: FILTERED VAL SET")
print("=" * 60)
val_f1_filt, preds_filt, labels_filt = evaluate(model_eval, val_loader_filt, DEVICE)
report_filt = get_detailed_report(labels_filt, preds_filt, LABEL_LIST)
print(f"Macro-F1: {val_f1_filt:.4f}\n")
print(classification_report(labels_filt, preds_filt,
                            labels=list(range(NUM_CLASSES)),
                            target_names=LABEL_LIST, zero_division=0))

# ── Unfiltered val evaluation ──
val_dataset_unfilt = PunctuationDataset(val_unfiltered, vocab, MAX_SEQ_LEN)
val_loader_unfilt = DataLoader(val_dataset_unfilt, batch_size=BS, shuffle=False,
                               collate_fn=collate_fn, num_workers=0)

print("\n" + "=" * 60)
print("EVALUATION: UNFILTERED VAL SET")
print("=" * 60)
val_f1_unfilt, preds_unfilt, labels_unfilt = evaluate(model_eval, val_loader_unfilt, DEVICE)
report_unfilt = get_detailed_report(labels_unfilt, preds_unfilt, LABEL_LIST)
print(f"Macro-F1: {val_f1_unfilt:.4f}\n")
print(classification_report(labels_unfilt, preds_unfilt,
                            labels=list(range(NUM_CLASSES)),
                            target_names=LABEL_LIST, zero_division=0))


EVALUATION: FILTERED VAL SET
Macro-F1: 0.5182

              precision    recall  f1-score   support

           O       0.98      0.97      0.98     83486
 COMMA_AFTER       0.37      0.48      0.42      1113
  BUTH_AFTER       0.40      0.58      0.47      1959
REMOVE_COMMA       0.14      0.40      0.21       112

    accuracy                           0.95     86670
   macro avg       0.47      0.61      0.52     86670
weighted avg       0.96      0.95      0.96     86670


EVALUATION: UNFILTERED VAL SET
Macro-F1: 0.4104

              precision    recall  f1-score   support

           O       0.99      0.97      0.98    241960
 COMMA_AFTER       0.18      0.48      0.27      1113
  BUTH_AFTER       0.18      0.58      0.27      1959
REMOVE_COMMA       0.07      0.41      0.12       112

    accuracy                           0.96    245144
   macro avg       0.36      0.61      0.41    245144
weighted avg       0.98      0.96      0.97    245144



## Insertion-Only F1 (COMMA_AFTER + BUTH_AFTER average)

In [46]:
def compute_insertion_f1(report):
    '''Average F1 of insertion classes only (COMMA_AFTER + BUTH_AFTER).'''
    comma_f1 = report.get("COMMA_AFTER", {}).get("f1-score", 0)
    buth_f1 = report.get("BUTH_AFTER", {}).get("f1-score", 0)
    return (comma_f1 + buth_f1) / 2

ins_f1_filt = compute_insertion_f1(report_filt)
ins_f1_unfilt = compute_insertion_f1(report_unfilt)

print(f"Insertion-only F1 (filtered):   {ins_f1_filt:.4f}")
print(f"Insertion-only F1 (unfiltered): {ins_f1_unfilt:.4f}")


Insertion-only F1 (filtered):   0.4458
Insertion-only F1 (unfiltered): 0.2708


## Save Results

In [51]:
# Updating checkpoint with training history
checkpoint["training_history"] = history
torch.save(checkpoint, CHECKPOINT_PATH)

# Saving results JSON
results = {
    "model": "BiLSTM (buth-fixed)",
    "tokenizer_version": "buth_fixed",
    "num_classes": NUM_CLASSES,
    "label_map": LABEL_MAP,
    "best_params": best_params,
    "best_epoch": best_epoch,
    "vocab_size": VOCAB_SIZE,
    "training_history": history,
    "filtered_val": {
        "macro_f1": val_f1_filt,
        "insertion_f1": ins_f1_filt,
        "per_class": {name: {"precision": report_filt[name]["precision"],
                            "recall": report_filt[name]["recall"],
                            "f1": report_filt[name]["f1-score"],
                            "support": report_filt[name]["support"]} for name in LABEL_LIST}},
    "unfiltered_val": {"macro_f1": val_f1_unfilt,
                    "insertion_f1": ins_f1_unfilt,
                    "per_class": {name: {
                        "precision": report_unfilt[name]["precision"],
                        "recall": report_unfilt[name]["recall"],
                        "f1": report_unfilt[name]["f1-score"],
                        "support": report_unfilt[name]["support"]} for name in LABEL_LIST}},
    "label_distribution": {
        "train_filtered": {LABEL_LIST[k]: v for k, v in train_filt_counts.items()},
        "val_filtered": {LABEL_LIST[k]: v for k, v in val_filt_counts.items()},
        "val_unfiltered": {LABEL_LIST[k]: v for k, v in val_lbl_counts.items()}},
            "timestamp": datetime.now().isoformat()}
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.bool_,)):
            return bool(obj)
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

with open(RESULTS_PATH, 'w') as f:
    json.dump(results, f, indent=2, cls=NumpyEncoder)

print(f"Results saved to {RESULTS_PATH}")
print(f"Checkpoint saved to {CHECKPOINT_PATH}")


Results saved to Weights - train_bilstm_v3/bilstm_v3_results.json
Checkpoint saved to Weights - train_bilstm_v3/bilstm_v3_best.pt


## Phase 3: Cross-Model Comparison Table

In [54]:
# BiLSTM results
v3_filt = val_f1_filt
v3_unfilt = val_f1_unfilt
v3_ins_filt = ins_f1_filt
v3_ins_unfilt = ins_f1_unfilt

# Per-class for v3
v3_comma_filt = report_filt["COMMA_AFTER"]
v3_buth_filt = report_filt["BUTH_AFTER"]
v3_remove_filt = report_filt["REMOVE_COMMA"]
v3_comma_unfilt = report_unfilt["COMMA_AFTER"]
v3_buth_unfilt = report_unfilt["BUTH_AFTER"]
v3_remove_unfilt = report_unfilt["REMOVE_COMMA"]

print("=" * 80)
print("CROSS-MODEL COMPARISON (all with buth-fixed tokenizer)")
print("=" * 80)
print(f"{'Model':<30} {'Cls':>3} {'Val Set':<12} {'Macro-F1':>9} {'Ins F1':>8}")
print("-" * 80)
print(f"{'BiLSTM v2 (OLD, buggy)':<30} {'4':>3} {'Filtered':<12} {'0.7206':>9} {'—':>8}")
print(f"{'BiLSTM v2 (OLD, buggy)':<30} {'4':>3} {'Unfiltered':<12} {'0.5933':>9} {'0.3860':>8}")
print(f"{'BiLSTM':<30} {'4':>3} {'Filtered':<12} {v3_filt:>9.4f} {v3_ins_filt:>8.4f}")
print(f"{'BiLSTM':<30} {'4':>3} {'Unfiltered':<12} {v3_unfilt:>9.4f} {v3_ins_unfilt:>8.4f}")
print(f"{'HyeBERT':<30} {'4':>3} {'Unfiltered':<12} {'0.4110':>9} {'—':>8}")
print(f"{'mBERT (if done)':<30} {'4':>3} {'Unfiltered':<12} {'???':>9} {'???':>8}")
print()

print("=" * 80)
print("PER-CLASS DETAIL — BiLSTM")
print("=" * 80)
print(f"\nFiltered val (macro-F1={v3_filt:.4f}):")
print(f"  COMMA_AFTER:  P={v3_comma_filt['precision']:.3f}  R={v3_comma_filt['recall']:.3f}  F1={v3_comma_filt['f1-score']:.3f}  (n={v3_comma_filt['support']})")
print(f"  BUTH_AFTER:   P={v3_buth_filt['precision']:.3f}  R={v3_buth_filt['recall']:.3f}  F1={v3_buth_filt['f1-score']:.3f}  (n={v3_buth_filt['support']})")
print(f"  REMOVE_COMMA: P={v3_remove_filt['precision']:.3f}  R={v3_remove_filt['recall']:.3f}  F1={v3_remove_filt['f1-score']:.3f}  (n={v3_remove_filt['support']})")

print(f"\nUnfiltered val (macro-F1={v3_unfilt:.4f}):")
print(f"  COMMA_AFTER:  P={v3_comma_unfilt['precision']:.3f}  R={v3_comma_unfilt['recall']:.3f}  F1={v3_comma_unfilt['f1-score']:.3f}  (n={v3_comma_unfilt['support']})")
print(f"  BUTH_AFTER:   P={v3_buth_unfilt['precision']:.3f}  R={v3_buth_unfilt['recall']:.3f}  F1={v3_buth_unfilt['f1-score']:.3f}  (n={v3_buth_unfilt['support']})")
print(f"  REMOVE_COMMA: P={v3_remove_unfilt['precision']:.3f}  R={v3_remove_unfilt['recall']:.3f}  F1={v3_remove_unfilt['f1-score']:.3f}  (n={v3_remove_unfilt['support']})")


CROSS-MODEL COMPARISON (all with buth-fixed tokenizer)
Model                          Cls Val Set       Macro-F1   Ins F1
--------------------------------------------------------------------------------
BiLSTM v2 (OLD, buggy)           4 Filtered        0.7206        —
BiLSTM v2 (OLD, buggy)           4 Unfiltered      0.5933   0.3860
BiLSTM v3 (buth-fixed)           4 Filtered        0.5182   0.4458
BiLSTM v3 (buth-fixed)           4 Unfiltered      0.4104   0.2708
HyeBERT v3                       4 Unfiltered      0.4110        —
mBERT (if done)                  4 Unfiltered         ???      ???

PER-CLASS DETAIL — BiLSTM v3 (buth-fixed)

Filtered val (macro-F1=0.5182):
  COMMA_AFTER:  P=0.372  R=0.477  F1=0.418  (n=1113.0)
  BUTH_AFTER:   P=0.401  R=0.578  F1=0.474  (n=1959.0)
  REMOVE_COMMA: P=0.138  R=0.402  F1=0.205  (n=112.0)

Unfiltered val (macro-F1=0.4104):
  COMMA_AFTER:  P=0.185  R=0.480  F1=0.267  (n=1113.0)
  BUTH_AFTER:   P=0.180  R=0.577  F1=0.275  (n=1959.0)
  REMOVE_C

## Summary

**Outputs produced:**
1. `armenian_vocab.json` — new vocabulary
2. `armenian_embeddings.pt` — new embeddings (reusing old GloVe vectors where possible)
3. `bilstm_v3_best.pt` — trained checkpoint (dict format, compatible with v2 loading code)
4. `bilstm_v3_results.json` — complete evaluation metrics